# Taxonomy Coverage Validation (Step 4)

Check whether the governed 104-leaf billing taxonomy (`code/artifacts/taxonomy.json`) is *representatively good enough* to cover the reprocessed billing/payments narratives.

**Method:** 20% random sample → one structured narrative per call
`Customer Intent Tag | Problem Identified Tag | Reason | Summary` → embedded with **BAAI/bge-large-en-v1.5** → matched to the 104 leaf prototypes (each = mean of its 10 curated centroid embeddings) via cosine similarity. Covered when max-sim ≥ 0.30.

In [ ]:
import json, pandas as pd
OUT='code/output'
try:
    rep = json.load(open(f'{OUT}/coverage_report.json'))
except FileNotFoundError:
    import sys; sys.path.insert(0,'code/pipeline')
    import coverage as C
    rep = C.run(frac=0.20, threshold=0.30, device='cuda')  # may take ~20-30min on GPU
    json.dump(rep, open(f'{OUT}/coverage_report.json','w'), indent=2)
for k in ('n_samples','n_covered','n_fallback','coverage_pct','fallback_pct',
          'mean_best_sim','median_best_sim'):
    print(f'{k:14}: {rep[k]}')

## Coverage by macro category

In [ ]:
cat = pd.DataFrame(rep['category_coverage'])
print(cat.sort_values('calls', ascending=False).to_string(index=False))

## Coverage by leaf (top volume)

In [ ]:
leaf = pd.DataFrame(rep['leaf_coverage']).sort_values('calls', ascending=False)
print('\n'.join('%-42s %7d calls  %5.1f%% covered' % (r['canonical_tag'], r['calls'], r['coverage_pct'])
               for _, r in leaf.head(15).iterrows()))
print('\n... lowest coverage leaves:')
low = leaf[leaf['calls']>=20].sort_values('coverage_pct').head(10)
print('\n'.join('%-42s %7d calls  %5.1f%% covered' % (r['canonical_tag'], r['calls'], r['coverage_pct'])
               for _, r in low.iterrows()))

## Review the most-uncovered narratives

In [ ]:
pd.DataFrame(rep['uncovered_review'])[['record_id','Call_Month','_sim','_tag','_cat','narrative']]

## Conclusion
If `coverage_pct` is high (e.g. ≥ 90%) and the uncovered sample shows genuine billing reasons already represented by existing leaves, the taxonomy is **sufficient and kept unchanged**. If low-coverage leaves reveal recurring unmatched billing reasons, propose new canonical leaves through the governance loop.